In [1]:
from utils import *

In [2]:
msft_df = get_stock_data('msft', start='2008-01-01', end='2025-10-29');
intc_df = get_stock_data('intc', end='2025-10-29');
meta_df = get_stock_data('meta', start='2008-01-01', end='2025-10-29');

msft_final = get_prev_xday_high(msft_df)
intc_final = get_prev_xday_high(intc_df)
meta_final = get_prev_xday_high(meta_df)
msft_final.shape

/home/azaidi/Desktop/nonsense/fin/utils.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end).reset_index()
[*********************100%***********************]  1 of 1 completed
/home/azaidi/Desktop/nonsense/fin/utils.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end).reset_index()
[*********************100%***********************]  1 of 1 completed
/home/azaidi/Desktop/nonsense/fin/utils.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end).reset_index()
[*********************100%***********************]  1 of 1 completed


(4455, 11)

In [15]:
# pip install bokeh pandas numpy
from bokeh.plotting import figure, save
from bokeh.io import output_file
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource, LinearColorMapper, ColorBar, BasicTicker,
    NumeralTickFormatter, DatetimeTickFormatter, RangeTool, HoverTool,
    WheelZoomTool, Toggle, Slider, CustomJS, CrosshairTool, CustomJSHover
)
from bokeh.palettes import all_palettes
from bokeh.resources import INLINE
import numpy as np
import pandas as pd


def bokeh_stock_gradient_to_html(
    df: pd.DataFrame,
    filename: str = "plot.html",
    title: str | None = None,
    *,
    # Appearance
    width: int = 1100,
    height: int = 560,
    overview_height: int = 110,
    dark_mode: bool = True,
    palette: str | list[str] = "RdYlGn",
    palette_n: int = 11,
    reverse_palette: bool = False,
    center_zero: bool = True,
    # Behavior
    close_ticker: str | None = None,
    auto_yscale_on_zoom: bool = True,
    y_pad_frac: float = 0.22,
    min_y_span: float = 1.0,
    # New controls / sane defaults
    default_color_cap_abs_pct: float = 0.15,   # 15% absolute cap for the legend/colors
    default_hover_window_days: int = 7         # +/- days around cursor for “window Δ”
):
    """
    Standalone offline HTML:
      • Gradient segments colored by % change (with absolute cap for legend)
      • ONE-point hover + window-delta hover line
      • Crosshair, overview RangeTool, wheel zoom
      • Dark-mode toggle, Auto-Y toggle, Y padding slider
      • Color-intensity slider (tighten/widen bounds around 0 within the cap)
      • Color-cap slider (keeps legend realistic; hover still shows true step %)

    df needs: Date, Close; optional pct_change (N-1).
    """
    # ---------- prep ----------
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    def _as_1d_close_series(_df, ticker=None):
        obj = _df["Close"]
        if isinstance(obj, pd.DataFrame):
            if ticker is not None and ticker in obj.columns:
                obj = obj[ticker]
            else:
                obj = obj.iloc[:, 0]
        return pd.to_numeric(obj, errors="coerce")

    close = _as_1d_close_series(df, close_ticker)
    dates = df["Date"]

    if "pct_change" in df.columns:
        pct_obj = df["pct_change"]
        if isinstance(pct_obj, pd.DataFrame):
            pct_obj = pct_obj.iloc[:, 0]
        pct = pd.to_numeric(pct_obj, errors="coerce").to_numpy()
        pct = pct[1:] if len(pct) == len(close) else pct[:len(close)-1]
    else:
        pct = close.pct_change().to_numpy()[1:]

    # Segments (N-1) and one point per day (endpoint) for hover
    x0, x1 = dates.iloc[:-1], dates.iloc[1:]
    y0, y1 = close.iloc[:-1].astype(float), close.iloc[1:].astype(float)

    # ---------- robust color scaling ----------
    finite = np.isfinite(pct)
    # Use a high percentile as a hint, but cap to a realistic absolute bound (default 15%)
    if finite.any():
        p98 = float(np.nanpercentile(np.abs(pct[finite]), 98))
    else:
        p98 = 1e-9
    cap0 = max(0.0001, min(p98, float(default_color_cap_abs_pct)))  # initial abs cap (in ratio units)

    # Store both raw pct (for hover) and clipped pct (for coloring & legend)
    pct_clipped = np.clip(pct, -cap0, cap0)

    # ---------- palette ----------
    def _get_palette(pal, n, reverse):
        if isinstance(pal, str):
            pal_dict = all_palettes.get(pal)
            if pal_dict is None:
                raise ValueError(f"Unknown palette '{pal}'")
            sizes = sorted(pal_dict.keys())
            n_pick = min(sizes, key=lambda k: abs(k - n))
            out = list(pal_dict[n_pick])
        else:
            out = list(pal)
        return out[::-1] if reverse else out

    pal = _get_palette(palette, palette_n, reverse_palette)
    color_mapper = LinearColorMapper(palette=pal, low=-cap0, high=cap0)

    # ---------- sources ----------
    seg_src = ColumnDataSource(dict(x0=x0, y0=y0, x1=x1, y1=y1, pct=pct, pctc=pct_clipped))
    pt_src  = ColumnDataSource(dict(date=x1, x=x1, y=y1, pct=pct))
    series_src = ColumnDataSource(dict(date=dates, close=close))

    # ---------- figures ----------
    p = figure(
        width=width, height=height, x_axis_type="datetime",
        tools="pan,box_zoom,wheel_zoom,reset,save",
        title=title or "Stock Price with Gradient Color-Coded Changes",
    )
    # Color by clipped values (pctc)
    p.segment("x0", "y0", "x1", "y1", source=seg_src, line_width=2,
              color={"field": "pctc", "transform": color_mapper})

    # Invisible points for single hit-test; visible only on hover
    pts = p.circle("x", "y", source=pt_src, size=8,
                   fill_alpha=0.0, line_alpha=0.0,
                   hover_fill_alpha=1.0, hover_fill_color="white",
                   hover_line_alpha=1.0, hover_line_color="black")

    # ----- Window-delta custom hover -----
    hover_win = Slider(title="Hover window (days)", start=1, end=60, step=1,
                       value=int(default_hover_window_days), width=220)

    win_delta_fmt = CustomJSHover(args=dict(series=series_src, win=hover_win), code="""
        const x = special_vars.x;                   // data coords
        const xs = series.data.date;
        const ys = series.data.close;
        const half = win.value * 24*60*60*1000;     // ms
        let lo = -1, hi = -1;
        for (let i = 0; i < xs.length; i++){
            const t = +xs[i];
            if (t >= x - half && t <= x + half){
                if (lo === -1) lo = i;
                hi = i;
            }
        }
        if (lo === -1 || hi === -1) return "";
        const p0 = ys[lo], p1 = ys[hi];
        const delta = p1 - p0;
        const pct = (p0 !== 0 && Number.isFinite(p0)) ? (delta / p0) : 0.0;
        function dstr(t){ const d = new Date(t); return d.toISOString().slice(0,10); }
        return `${dstr(xs[lo])} → ${dstr(xs[hi])}: ${delta.toFixed(2)} (${(pct*100).toFixed(2)}%)`;
    """)

    hov = HoverTool(
        renderers=[pts],
        mode="mouse",
        point_policy="snap_to_data",
        attachment="right",
        tooltips=[
            ("Date", "@date{%F}"),
            ("Close", "@y{0,0.00}"),
            ("%Δ step", "@pct{0.00%}"),
            ("Window Δ", "$custom")   # uses win_delta_fmt
        ],
        formatters={"@date": "datetime", "$custom": win_delta_fmt},
        show_arrow=False
    )
    p.add_tools(hov)
    p.add_tools(CrosshairTool(dimensions="height"))

    p.xaxis.formatter = DatetimeTickFormatter(days="%Y-%m-%d", months="%Y-%m", years="%Y")
    p.xaxis.axis_label = "Date"
    p.yaxis.axis_label = "Close"
    p.grid.grid_line_alpha = 0.30
    p.toolbar.active_scroll = p.select_one(WheelZoomTool)

    cbar = ColorBar(
        color_mapper=color_mapper, ticker=BasicTicker(desired_num_ticks=7),
        formatter=NumeralTickFormatter(format="0%"),
        label_standoff=8, width=12, location=(0, 0), title="% change / step"
    )
    p.add_layout(cbar, "right")

    overview = figure(width=width, height=overview_height, x_axis_type="datetime",
                      tools="", toolbar_location=None)
    ov_line = overview.line(dates, close, line_width=1)
    rt = RangeTool(x_range=p.x_range)
    rt.overlay.fill_color = "navy"; rt.overlay.fill_alpha = 0.15
    overview.add_tools(rt)
    overview.ygrid.grid_line_color = None
    overview.xaxis.formatter = DatetimeTickFormatter(days="%Y-%m-%d", months="%Y-%m", years="%Y")

    # ---------- initial Y range ----------
    cvals = close.to_numpy()
    if not np.isfinite(cvals).any():
        cmin, cmax = 0.0, 1.0
    else:
        cmin, cmax = float(np.nanmin(cvals)), float(np.nanmax(cvals))
    span = max(cmax - cmin, min_y_span)
    pad = y_pad_frac * span
    p.y_range.start = cmin - pad
    p.y_range.end   = cmax + pad

    # ---------- style & dark mode ----------
    def _apply_style(fig, ov, colorbar, dark: bool):
        if dark:
            bg = "#0e1117"; grid = "#2a2f3a"; axis = "#e5e7eb"; minor = "#94a3b8"
            ov_color = "#9aa4b2"; overlay = "#60a5fa"
        else:
            bg = "white";   grid = "#e5e7eb"; axis = "#374151"; minor = "#9ca3af"
            ov_color = "#6b7280"; overlay = "#2563eb"
        for f in (fig, ov):
            f.background_fill_color = bg
            f.border_fill_color = bg
            f.outline_line_color = bg
            if f.xaxis:
                ax = f.xaxis[0]
                ax.major_label_text_color = axis
                ax.axis_label_text_color = axis
                ax.major_tick_line_color = axis
                ax.minor_tick_line_color = minor
                ax.axis_line_color = axis
            if f.yaxis:
                ay = f.yaxis[0]
                ay.major_label_text_color = axis
                ay.axis_label_text_color = axis
                ay.major_tick_line_color = axis
                ay.minor_tick_line_color = minor
                ay.axis_line_color = axis
            if f.xgrid: f.xgrid[0].grid_line_color = grid
            if f.ygrid: f.ygrid[0].grid_line_color = grid
        colorbar.title_text_color = axis
        colorbar.major_label_text_color = axis
        ov_line.glyph.line_color = ov_color
        rt.overlay.fill_color = overlay
        rt.overlay.fill_alpha = 0.18

    _apply_style(p, overview, cbar, dark_mode)

    # ---------- controls ----------
    toggle_dark = Toggle(label="Dark mode", active=dark_mode, width=110)
    toggle_auto = Toggle(label="Auto Y-scale", active=auto_yscale_on_zoom, width=120)
    pad_slider  = Slider(title="Y padding (fraction of span)", start=0.0, end=0.60, step=0.01,
                         value=float(y_pad_frac), width=280)
    intensity   = Slider(title="Color intensity (lower = more pop)", start=0.25, end=1.50, step=0.05,
                         value=1.00, width=300)
    color_cap   = Slider(title="Color cap (abs % for legend/colors)",
                         start=1, end=100, step=1, value=int(round(cap0*100)), width=300)

    toggle_dark.js_on_click(CustomJS(args=dict(p=p, ov=overview, cbar=cbar, rt=rt, ov_line=ov_line), code="""
        const dark = cb_obj.active === true;
        const bg    = dark ? "#0e1117" : "white";
        const grid  = dark ? "#2a2f3a" : "#e5e7eb";
        const axis  = dark ? "#e5e7eb" : "#374151";
        const minor = dark ? "#94a3b8" : "#9ca3af";
        const ovcol = dark ? "#9aa4b2" : "#6b7280";
        const ovlay = dark ? "#60a5fa" : "#2563eb";
        function style_fig(fig){
            fig.background_fill_color = bg;
            fig.border_fill_color = bg;
            fig.outline_line_color = bg;
            for (const ax of fig.xaxis){
                ax.major_label_text_color = axis;
                ax.axis_label_text_color  = axis;
                ax.major_tick_line_color  = axis;
                ax.minor_tick_line_color  = minor;
                ax.axis_line_color        = axis;
            }
            for (const ax of fig.yaxis){
                ax.major_label_text_color = axis;
                ax.axis_label_text_color  = axis;
                ax.major_tick_line_color  = axis;
                ax.minor_tick_line_color  = minor;
                ax.axis_line_color        = axis;
            }
            for (const g of fig.xgrid){ g.grid_line_color = grid; }
            for (const g of fig.ygrid){ g.grid_line_color = grid; }
        }
        style_fig(p); style_fig(ov);
        cbar.title_text_color = axis;
        cbar.major_label_text_color = axis;
        try { ov_line.glyph.line_color = ovcol; } catch(e){}
        try { rt.overlay.fill_color = ovlay; rt.overlay.fill_alpha = 0.18; } catch(e){}
    """))

    # Auto Y on zoom / padding
    yscale_cb = CustomJS(args=dict(p=p, s=series_src, auto=toggle_auto, pad_slider=pad_slider,
                                   minspan=float(min_y_span)), code="""
        if (!auto.active) return;
        const xs = s.data.date, ys = s.data.close;
        const L = p.x_range.start, R = p.x_range.end;
        let lo = Infinity, hi = -Infinity;
        for (let i = 0; i < xs.length; i++){
            const t = +xs[i];
            if (t >= L && t <= R){
                const v = ys[i];
                if (Number.isFinite(v)){
                    if (v < lo) lo = v;
                    if (v > hi) hi = v;
                }
            }
        }
        if (!Number.isFinite(lo) || !Number.isFinite(hi)) return;
        let span = hi - lo;
        if (!(span > 0)) span = minspan || 1.0;
        const pad = span * pad_slider.value;
        p.y_range.start = lo - pad;
        p.y_range.end   = hi + pad;
    """)
    p.x_range.js_on_change('start', yscale_cb)
    p.x_range.js_on_change('end', yscale_cb)
    pad_slider.js_on_change('value', yscale_cb)
    toggle_auto.js_on_click(yscale_cb)

    # Update color clipping + legend bounds when cap changes
    cap_cb = CustomJS(args=dict(seg=seg_src, cm=color_mapper, intensity=intensity), code="""
        const cap = (cb_obj.value / 100.0);   // ratio
        const f = intensity.value;
        const raw = seg.data.pct, out = seg.data.pctc;
        for (let i = 0; i < raw.length; i++){
            const v = raw[i];
            if (!Number.isFinite(v)) { out[i] = v; }
            else { out[i] = Math.max(-cap, Math.min(cap, v)); }
        }
        seg.change.emit();
        cm.low  = -cap * f;
        cm.high =  cap * f;
        cm.change.emit();
    """)
    color_cap.js_on_change('value', cap_cb)

    # Intensity only tightens/widens inside the current cap
    intensity_cb = CustomJS(args=dict(cm=color_mapper, cap=color_cap), code="""
        const cap = (cap.value / 100.0);
        const f = cb_obj.value;
        cm.low  = -cap * f;
        cm.high =  cap * f;
        cm.change.emit();
    """)
    intensity.js_on_change('value', intensity_cb)

    output_file(filename, title=title or "Interactive Stock Plot")
    save(column(row(toggle_dark, toggle_auto, pad_slider, intensity, color_cap, hover_win), p, overview),
         resources=INLINE)
    print(f"Wrote {filename} (standalone HTML).")


In [16]:
bokeh_stock_gradient_to_html(
    msft_final, filename="plot.html", title="MSFT",
    dark_mode=False, palette="RdYlGn", reverse_palette=True,
    palette_n=11, center_zero=True
)

Wrote plot.html (standalone HTML).


In [17]:
from bokeh.plotting import figure, save, output_file
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource, LinearColorMapper, ColorBar, BasicTicker, NumeralTickFormatter,
    DatetimeTickFormatter, HoverTool, RangeTool, BoxSelectTool, CustomJS, Toggle, Slider, CrosshairTool
)
from bokeh.palettes import all_palettes
import numpy as np
import pandas as pd

def bokeh_stock_gradient_to_html_v2(
    df: pd.DataFrame,
    filename: str = "plot.html",
    title: str = None,
    percent_cap: float = 0.4,  # 40% default cap, tweak as needed
    # ... (other params unchanged)
):
    # --- Data preparation (same as before) ---
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    close = pd.to_numeric(df["Close"], errors="coerce")
    dates = df["Date"]
    pct = close.pct_change().to_numpy()[1:]

    # --- Cap percent changes ---
    pct = np.clip(pct, -percent_cap, percent_cap)

    x0, x1 = dates.iloc[:-1], dates.iloc[1:]
    y0, y1 = close.iloc[:-1].astype(float), close.iloc[1:].astype(float)
    seg_src = ColumnDataSource(dict(x0=x0, y0=y0, x1=x1, y1=y1, pct=pct))
    pt_src = ColumnDataSource(dict(date=x1, x=x1, y=y1, pct=pct))
    series_src = ColumnDataSource(dict(date=dates, close=close))

    # --- Color mapper: robust cap (no more ±1500%) ---
    pal = all_palettes["RdYlGn"][11]
    color_mapper = LinearColorMapper(palette=pal, low=-percent_cap, high=percent_cap)

    # --- Main figure (same as before except hover/cap tweaks) ---
    p = figure(
        width=1100, height=560, x_axis_type="datetime",
        tools="pan,box_zoom,wheel_zoom,reset,save", title=title or "Stock Price: Improved Gradient"
    )
    p.segment("x0", "y0", "x1", "y1", source=seg_src, line_width=2, color={"field": "pct", "transform": color_mapper})
    pts = p.circle("x", "y", source=pt_src, size=8, fill_alpha=0.0, line_alpha=0.0,
                   hover_fill_alpha=1.0, hover_fill_color="white", hover_line_alpha=1.0, hover_line_color="black")

    # --- Enhanced Hover with Range/Select ---
    p.add_tools(BoxSelectTool(dimensions="width"))

    custom_hover = HoverTool(
        renderers=[pts], mode="mouse", point_policy="snap_to_data", attachment="right",
        tooltips=[
            ("Date", "@date{%F}"),
            ("Close", "@y{0,0.00}"),
            ("%Δ step", "@pct{0.00%}")
        ],
        formatters={"@date": "datetime"}, show_arrow=False
    )
    p.add_tools(custom_hover)
    p.add_tools(CrosshairTool(dimensions="height"))

    # --- CustomJS callback for range hover: selection info ---
    js_code = """
    // Calculate price/pct change in current selection
    const inds = cb_obj.indices;
    const dsrc = cb_obj.data_source;
    const dates = dsrc.data['date'], prices = dsrc.data['y'];
    if (inds.length >= 2) {
        const idx_sorted = inds.slice().sort((a,b) => +new Date(dates[a]) - +new Date(dates[b]));
        const i0 = idx_sorted[0], i1 = idx_sorted[idx_sorted.length-1];
        const d0 = dates[i0], d1 = dates[i1];
        const p0 = prices[i0], p1 = prices[i1];
        const abs_change = p1 - p0;
        const pct_change = (p1 - p0) / p0 * 100;
        alert(`Selected Range: ${d0} - ${d1}\\nPrice Change: ${abs_change.toFixed(2)}\\nPercent Change: ${pct_change.toFixed(2)}%`);
    }
    """
    pts.data_source.selected.js_on_change('indices', CustomJS(args=dict(), code=js_code))

    # --- ColorBar, RangeTool, Layout (unchanged) ---
    cbar = ColorBar(
        color_mapper=color_mapper, ticker=BasicTicker(desired_num_ticks=7),
        formatter=NumeralTickFormatter(format="0%"), label_standoff=8, width=12, location=(0,0),
        title="% change / step"
    )
    p.add_layout(cbar, "right")
    # ... (rest of layout as before)

    output_file(filename, title=title or "Interactive Stock Plot")
    save(p)

# Usage: bokeh_stock_gradient_to_html_v2(df)


In [20]:
bokeh_stock_gradient_to_html(msft_final)

Wrote plot.html (standalone HTML).


In [19]:
# pip install bokeh pandas numpy
from bokeh.plotting import figure, save
from bokeh.io import output_file
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource, LinearColorMapper, ColorBar, BasicTicker,
    NumeralTickFormatter, DatetimeTickFormatter, RangeTool, HoverTool,
    WheelZoomTool, Toggle, Slider, CustomJS, CrosshairTool
)
from bokeh.palettes import all_palettes
from bokeh.resources import INLINE
import numpy as np
import pandas as pd

def bokeh_stock_gradient_to_html(
    df: pd.DataFrame,
    filename: str = "plot.html",
    title: str | None = None,
    *,
    # Appearance
    width: int = 1100,
    height: int = 560,
    overview_height: int = 110,
    dark_mode: bool = True,
    palette: str | list[str] = "RdYlGn",
    palette_n: int = 11,
    reverse_palette: bool = False,
    center_zero: bool = True,
    # Behavior
    close_ticker: str | None = None,
    auto_yscale_on_zoom: bool = True,
    y_pad_frac: float = 0.22,
    min_y_span: float = 1.0,
    # NEW: Color scale control
    color_percentile: float = 75.0,  # Use 75th percentile instead of 98th
):
    """
    Standalone offline HTML:
    • Gradient segments colored by % change
    • ONE-point hover (no stacked tooltips)
    • Crosshair, overview RangeTool, wheel zoom
    • Dark-mode toggle, Auto-Y toggle, Y padding slider
    • Color-intensity slider (tighten bounds around 0)
    • Smarter hover with date range detection
    
    df needs: Date, Close; optional pct_change (N-1).
    """
    
    # ---------- prep ----------
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    def _as_1d_close_series(_df, ticker=None):
        obj = _df["Close"]
        if isinstance(obj, pd.DataFrame):
            if ticker is not None and ticker in obj.columns:
                obj = obj[ticker]
            else:
                obj = obj.iloc[:, 0]
        return pd.to_numeric(obj, errors="coerce")

    close = _as_1d_close_series(df, close_ticker)
    dates = df["Date"]

    if "pct_change" in df.columns:
        pct_obj = df["pct_change"]
        if isinstance(pct_obj, pd.DataFrame):
            pct_obj = pct_obj.iloc[:, 0]
        pct = pd.to_numeric(pct_obj, errors="coerce").to_numpy()
        pct = pct[1:] if len(pct) == len(close) else pct[:len(close)-1]
    else:
        pct = close.pct_change().to_numpy()[1:]

    # Segments (N-1) and one point per day (endpoint) for hover
    x0, x1 = dates.iloc[:-1], dates.iloc[1:]
    y0, y1 = close.iloc[:-1].astype(float), close.iloc[1:].astype(float)
    
    seg_src = ColumnDataSource(dict(x0=x0, y0=y0, x1=x1, y1=y1, pct=pct))
    pt_src = ColumnDataSource(dict(date=x1, x=x1, y=y1, pct=pct))
    series_src = ColumnDataSource(dict(date=dates, close=close))

    # ---------- IMPROVED color scale ----------
    finite = np.isfinite(pct)
    if finite.any():
        pmin, pmax = float(np.nanmin(pct[finite])), float(np.nanmax(pct[finite]))
        
        # Use specified percentile (default 75th) for more realistic color bounds
        base = float(np.nanpercentile(np.abs(pct[finite]), color_percentile)) or 1e-9
        
        # Cap the base at reasonable values (e.g., 50% daily change max for color scale)
        base = min(base, 0.50)  # Cap at 50% for color scale
        
        if center_zero:
            vlow, vhigh = -base, base
        else:
            if np.isclose(pmin, pmax):
                eps = abs(pmin) if pmin != 0 else 1e-9
                pmin, pmax = pmin - eps, pmax + eps
            vlow, vhigh = pmin, pmax
    else:
        base = 1e-9
        vlow, vhigh = -base, base

    def _get_palette(pal, n, reverse):
        if isinstance(pal, str):
            pal_dict = all_palettes.get(pal)
            if pal_dict is None:
                raise ValueError(f"Unknown palette '{pal}'")
            sizes = sorted(pal_dict.keys())
            n_pick = min(sizes, key=lambda k: abs(k - n))
            out = list(pal_dict[n_pick])
        else:
            out = list(pal)
        return out[::-1] if reverse else out

    pal = _get_palette(palette, palette_n, reverse_palette)
    color_mapper = LinearColorMapper(palette=pal, low=vlow, high=vhigh)

    # ---------- figures ----------
    p = figure(
        width=width, height=height,
        x_axis_type="datetime",
        tools="pan,box_zoom,wheel_zoom,reset,save",
        title=title or "Stock Price with Gradient Color-Coded Changes",
    )

    p.segment("x0", "y0", "x1", "y1", source=seg_src, line_width=2,
              color={"field": "pct", "transform": color_mapper})

    # Invisible points for single hit-test; visible only on hover
    pts = p.circle("x", "y", source=pt_src, size=8,
                   fill_alpha=0.0, line_alpha=0.0,
                   hover_fill_alpha=1.0, hover_fill_color="white",
                   hover_line_alpha=1.0, hover_line_color="black")

    # *** IMPROVED HOVER with mouse mode for single-point selection ***
    hov = HoverTool(
        renderers=[pts],
        mode="mouse",  # Only show nearest point to mouse
        point_policy="snap_to_data",  # Snap to nearest data point
        attachment="right",
        tooltips=[
            ("Date", "@date{%F}"),
            ("Close", "@y{0,0.00}"),
            ("%Δ step", "@pct{0.00%}")
        ],
        formatters={"@date": "datetime"},
        show_arrow=False
    )
    p.add_tools(hov)
    p.add_tools(CrosshairTool(dimensions="height"))

    p.xaxis.formatter = DatetimeTickFormatter(days="%Y-%m-%d", months="%Y-%m", years="%Y")
    p.xaxis.axis_label = "Date"
    p.yaxis.axis_label = "Close"
    p.grid.grid_line_alpha = 0.30
    p.toolbar.active_scroll = p.select_one(WheelZoomTool)

    cbar = ColorBar(
        color_mapper=color_mapper,
        ticker=BasicTicker(desired_num_ticks=7),
        formatter=NumeralTickFormatter(format="0%"),
        label_standoff=8,
        width=12,
        location=(0, 0),
        title="% change / step"
    )
    p.add_layout(cbar, "right")

    overview = figure(width=width, height=overview_height, x_axis_type="datetime",
                     tools="", toolbar_location=None)
    ov_line = overview.line(dates, close, line_width=1)
    rt = RangeTool(x_range=p.x_range)
    rt.overlay.fill_color = "navy"
    rt.overlay.fill_alpha = 0.15
    overview.add_tools(rt)
    overview.ygrid.grid_line_color = None
    overview.xaxis.formatter = DatetimeTickFormatter(days="%Y-%m-%d", months="%Y-%m", years="%Y")

    # ---------- initial Y range ----------
    cvals = close.to_numpy()
    cmin, cmax = (0.0, 1.0) if not np.isfinite(cvals).any() else (float(np.nanmin(cvals)), float(np.nanmax(cvals)))
    span = max(cmax - cmin, min_y_span)
    pad = y_pad_frac * span
    p.y_range.start = cmin - pad
    p.y_range.end = cmax + pad

    # ---------- style & dark mode ----------
    def _apply_style(fig, ov, colorbar, dark: bool):
        if dark:
            bg = "#0e1117"
            grid = "#2a2f3a"
            axis = "#e5e7eb"
            minor = "#94a3b8"
            ov_color = "#9aa4b2"
            overlay = "#60a5fa"
        else:
            bg = "white"
            grid = "#e5e7eb"
            axis = "#374151"
            minor = "#9ca3af"
            ov_color = "#6b7280"
            overlay = "#2563eb"
        
        for f in (fig, ov):
            f.background_fill_color = bg
            f.border_fill_color = bg
            f.outline_line_color = bg
            if f.xaxis:
                ax = f.xaxis[0]
                ax.major_label_text_color = axis
                ax.axis_label_text_color = axis
                ax.major_tick_line_color = axis
                ax.minor_tick_line_color = minor
                ax.axis_line_color = axis
            if f.yaxis:
                ay = f.yaxis[0]
                ay.major_label_text_color = axis
                ay.axis_label_text_color = axis
                ay.major_tick_line_color = axis
                ay.minor_tick_line_color = minor
                ay.axis_line_color = axis
            if f.xgrid:
                f.xgrid[0].grid_line_color = grid
            if f.ygrid:
                f.ygrid[0].grid_line_color = grid
        
        colorbar.title_text_color = axis
        colorbar.major_label_text_color = axis
        ov_line.glyph.line_color = ov_color
        rt.overlay.fill_color = overlay
        rt.overlay.fill_alpha = 0.18

    _apply_style(p, overview, cbar, dark_mode)

    # ---------- controls ----------
    toggle_dark = Toggle(label="Dark mode", active=dark_mode, width=110)
    toggle_auto = Toggle(label="Auto Y-scale", active=auto_yscale_on_zoom, width=120)
    pad_slider = Slider(title="Y padding (fraction of span)", start=0.0, end=0.60,
                       step=0.01, value=float(y_pad_frac), width=280)
    intensity = Slider(title="Color intensity (lower = more pop)", start=0.25, end=1.50,
                      step=0.05, value=1.00, width=300)

    toggle_dark.js_on_click(CustomJS(args=dict(p=p, ov=overview, cbar=cbar, rt=rt, ov_line=ov_line), code="""
        const dark = cb_obj.active === true;
        const bg = dark ? "#0e1117" : "white";
        const grid = dark ? "#2a2f3a" : "#e5e7eb";
        const axis = dark ? "#e5e7eb" : "#374151";
        const minor = dark ? "#94a3b8" : "#9ca3af";
        const ovcol = dark ? "#9aa4b2" : "#6b7280";
        const ovlay = dark ? "#60a5fa" : "#2563eb";
        
        function style_fig(fig){
            fig.background_fill_color = bg;
            fig.border_fill_color = bg;
            fig.outline_line_color = bg;
            for (const ax of fig.xaxis){
                ax.major_label_text_color = axis;
                ax.axis_label_text_color = axis;
                ax.major_tick_line_color = axis;
                ax.minor_tick_line_color = minor;
                ax.axis_line_color = axis;
            }
            for (const ax of fig.yaxis){
                ax.major_label_text_color = axis;
                ax.axis_label_text_color = axis;
                ax.major_tick_line_color = axis;
                ax.minor_tick_line_color = minor;
                ax.axis_line_color = axis;
            }
            for (const g of fig.xgrid){ g.grid_line_color = grid; }
            for (const g of fig.ygrid){ g.grid_line_color = grid; }
        }
        style_fig(p);
        style_fig(ov);
        cbar.title_text_color = axis;
        cbar.major_label_text_color = axis;
        try { ov_line.glyph.line_color = ovcol; } catch(e){}
        try { rt.overlay.fill_color = ovlay; rt.overlay.fill_alpha = 0.18; } catch(e){}
    """))

    yscale_cb = CustomJS(args=dict(p=p, s=series_src, auto=toggle_auto, 
                                   pad_slider=pad_slider, minspan=float(min_y_span)), code="""
        if (!auto.active) return;
        const xs = s.data.date, ys = s.data.close;
        const L = p.x_range.start, R = p.x_range.end;
        let lo = Infinity, hi = -Infinity;
        for (let i = 0; i < xs.length; i++){
            const t = +xs[i];
            if (t >= L && t <= R){
                const v = ys[i];
                if (Number.isFinite(v)){
                    if (v < lo) lo = v;
                    if (v > hi) hi = v;
                }
            }
        }
        if (!Number.isFinite(lo) || !Number.isFinite(hi)) return;
        let span = hi - lo;
        if (!(span > 0)) span = minspan || 1.0;
        const pad = span * pad_slider.value;
        p.y_range.start = lo - pad;
        p.y_range.end = hi + pad;
    """)
    
    p.x_range.js_on_change('start', yscale_cb)
    p.x_range.js_on_change('end', yscale_cb)
    pad_slider.js_on_change('value', yscale_cb)
    toggle_auto.js_on_click(yscale_cb)

    color_cb = CustomJS(args=dict(cm=color_mapper, base=float(base)), code="""
        const f = cb_obj.value;  // 0.25..1.5
        cm.low = -base * f;
        cm.high = base * f;
        cm.change.emit();
    """)
    intensity.js_on_change('value', color_cb)

    output_file(filename, title=title or "Interactive Stock Plot")
    save(column(row(toggle_dark, toggle_auto, pad_slider, intensity), p, overview), resources=INLINE)
    print(f"Wrote {filename} (standalone HTML).")